# Homework - Everything & Then Some

## Section 3 - Feature selection

### The golden rule of this section

In this section, we define, implement, and evaluate a structured feature-selection strategy using standard statistical and mathematical principles implemented directly with `numpy` and `pandas`.

We will pay critical attention to identifier-like variables to prevent data leakage and overfitting.

### 3.0 Setup & Baseline Data Prep

We start by importing only numpy, pandas, and matplotlib.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Random seed: makes the results reproducible
SEED = 42
np.random.seed(SEED)

# Name of the column we want to predict
TARGET = "repeat_purchase_90d"

train_clean = pd.read_csv("../hdata/train_clean.csv")

df_feat = train_clean.copy()

## 3.1 Identifier-like Variables & Critical Analysis

Identifier-like variables should be treated critically rather than accepted automatically. An identifier-like variable is any column that serves as a unique key, sequential order tracking index, or arbitrarily assigned grouping metric with high cardinality.

### Critical Identification and Decision Matrix

**`ID` & `customer_id` (Identifiers):**

- **Analysis:** ID represents a transaction key or row index. customer_id identifies the customer entity. Including ID directly in a model allows it to memorize sequence positions (spurious temporal leakage). While customer_id carries history, raw categorical IDs with high cardinality cause extreme overfitting and massive sparse expansion.

- **Decision:** Drop ID and customer_id completely from the predictive matrix $X$. Historical customer signals are already captured via engineered aggregate attributes (e.g., prior_purchase_count, prior_total_spend, customer_tenure_days).

**`random_noise` (Synthetic Artifact):**

- **Analysis:** Identified during initial EDA as noise. Including random noise serves as an empirical benchmark: any true predictor must rank higher in correlation/importance than random_noise.

- **Decision:** Keep briefly for baseline importance ranking tests, then drop before final modeling.

**High-Cardinality Categoricals (`transport_method`):**

- **Analysis:** Contains 80 distinct categories. Naïve encoding adds 80 sparse columns, inflating dimensionality relative to sample size without proportional predictive gain.

- **Decision:** Exclude high-cardinality text variables from numeric feature selection; retain low-cardinality indicators where justifiable.

In [2]:
# Separate target
y = df_feat[TARGET].astype(float)

# List identifiers and noise to remove from candidate feature space
identifiers_and_noise = ["ID", "customer_id", "random_noise", "purchase_date", "purchase_date_dt"]

# Candidate numerical feature matrix
candidate_cols = [
    c for c in df_feat.select_dtypes(include=["number"]).columns 
    if c not in [TARGET] + identifiers_and_noise
]

# Impute missing values with column medians for feature ranking algorithms
X_num = df_feat[candidate_cols].fillna(df_feat[candidate_cols].median())

print(f"Candidate feature space shape: {X_num.shape}")
print(f"Identifier variables dropped: {[c for c in ['ID', 'customer_id', 'random_noise'] if c in df_feat.columns]}")

Candidate feature space shape: (6469, 36)
Identifier variables dropped: ['ID', 'customer_id', 'random_noise']


## 3.2 Feature Selection Strategy (Pure Pandas & NumPy)
We implement a multi-tiered filter strategy using **Point-Biserial Correlation** and **ANOVA F-Score.**

### Step 1: Filter Method 1 — Absolute Point-Biserial Correlation
Measures the linear association between continuous numerical predictors and the binary target.

In [3]:
# Absolute linear correlation with target
correlations = X_num.apply(lambda col: col.corr(y)).abs()

filter_corr_df = pd.DataFrame({
    "feature": candidate_cols,
    "abs_corr": correlations
}).sort_values("abs_corr", ascending=False).reset_index(drop=True)

print("Top 10 Features by Absolute Correlation:")
print(filter_corr_df.head(10).to_string(index=False))

Top 10 Features by Absolute Correlation:
               feature  abs_corr
   purchase_count_365d  0.340343
   purchase_count_180d  0.334190
    purchase_count_90d  0.327921
            spend_180d  0.301013
  prior_purchase_count  0.298811
             spend_90d  0.268890
    purchase_count_30d  0.263339
     prior_total_spend  0.261934
         discount_rate  0.221933
purchase_rate_per_year  0.210093


### Step 2: Filter Method 2 — ANOVA F-Score (Signal-to-Noise Ratio)

Computes the ratio of between-group variance to within-group variance across target classes ($y=0$ vs $y=1$).$$\text{F-Score} = \frac{\text{Variance between groups}}{\text{Variance within groups}}$$

In [4]:
def compute_anova_f(col, y_vec):
    group_0 = col[y_vec == 0]
    group_1 = col[y_vec == 1]
    
    n_0, n_1 = len(group_0), len(group_1)
    mean_0, mean_1 = group_0.mean(), group_1.mean()
    overall_mean = col.mean()
    
    # Between-group variance
    ss_between = n_0 * (mean_0 - overall_mean)**2 + n_1 * (mean_1 - overall_mean)**2
    ms_between = ss_between / 1.0  # k - 1 = 2 - 1 = 1
    
    # Within-group variance
    ss_within = ((group_0 - mean_0)**2).sum() + ((group_1 - mean_1)**2).sum()
    ms_within = ss_within / (len(col) - 2)
    
    return ms_between / ms_within if ms_within > 0 else 0.0

f_scores = X_num.apply(lambda col: compute_anova_f(col, y))

filter_f_df = pd.DataFrame({
    "feature": candidate_cols,
    "f_score": f_scores
}).sort_values("f_score", ascending=False).reset_index(drop=True)

print("\nTop 10 Features by ANOVA F-Score:")
print(filter_f_df.head(10).to_string(index=False))


Top 10 Features by ANOVA F-Score:
               feature    f_score
   purchase_count_365d 847.231887
   purchase_count_180d 813.060894
    purchase_count_90d 779.201287
            spend_180d 644.349775
  prior_purchase_count 634.035975
             spend_90d 504.017774
    purchase_count_30d 481.887008
     prior_total_spend 476.383042
         discount_rate 335.029275
purchase_rate_per_year 298.629128


### Step 3: Empirical Benchmark Check against `random_noise`

We evaluate `random_noise` to confirm that selected features exceed random chance.

In [5]:
if "random_noise" in df_feat.columns:
    noise_col = df_feat["random_noise"].fillna(df_feat["random_noise"].median())
    noise_corr = abs(noise_col.corr(y))
    noise_f = compute_anova_f(noise_col, y)
    
    print(f"\nEmpirical Baseline - 'random_noise':")
    print(f"  - Abs Correlation: {noise_corr:.4f}")
    print(f"  - ANOVA F-Score:   {noise_f:.4f}")


Empirical Baseline - 'random_noise':
  - Abs Correlation: 0.0120
  - ANOVA F-Score:   0.9296


### Step 4: Redundancy & Collinearity Filtering

If two candidate features have a correlation above $0.85$, we keep the feature with the higher ANOVA F-score and drop the redundant variable.

In [6]:
# Compute pairwise correlation matrix among candidate features
corr_matrix = X_num.corr().abs()

# Identify collinear pairs (|r| > 0.85)
threshold = 0.85
to_drop_collinear = set()

for i in range(len(candidate_cols)):
    for j in range(i + 1, len(candidate_cols)):
        col1, col2 = candidate_cols[i], candidate_cols[j]
        if corr_matrix.loc[col1, col2] > threshold:
            # Compare F-scores to keep the superior feature
            if f_scores[col1] >= f_scores[col2]:
                to_drop_collinear.add(col2)
            else:
                to_drop_collinear.add(col1)

print(f"\nRedundant features removed due to high collinearity (> {threshold}):")
print(list(to_drop_collinear))


Redundant features removed due to high collinearity (> 0.85):
['recent_activity_ratio', 'spend_90d', 'Total Liquido', 'Total a Pagar', 'purchase_count_180d', 'purchase_count_90d']


### 3.3 Summary & Final Feature Set Justification

We aggregate rankings across filter metrics to select the final feature set.

In [7]:
# Combine Filter Scores
selection_matrix = filter_corr_df.merge(filter_f_df, on="feature")

# Remove collinear features
selection_matrix = selection_matrix[~selection_matrix["feature"].isin(to_drop_collinear)].copy()

# Compute Rank-Based Consensus
selection_matrix["rank_corr"] = selection_matrix["abs_corr"].rank(ascending=False)
selection_matrix["rank_f"] = selection_matrix["f_score"].rank(ascending=False)

selection_matrix["consensus_score"] = (selection_matrix["rank_corr"] + selection_matrix["rank_f"]) / 2.0
selection_matrix = selection_matrix.sort_values("consensus_score").reset_index(drop=True)

print("\n=== FEATURE SELECTION CONSENSUS MATRIX ===")
print(selection_matrix[["feature", "consensus_score", "abs_corr", "f_score"]].head(15).to_string(index=False))

# Extract Top 15 Features
FINAL_SELECTED_FEATURES = selection_matrix.head(15)["feature"].tolist()
print("\nFinal Selected Feature Set:")
print(FINAL_SELECTED_FEATURES)


=== FEATURE SELECTION CONSENSUS MATRIX ===
                     feature  consensus_score  abs_corr    f_score
         purchase_count_365d              1.0  0.340343 847.231887
                  spend_180d              2.0  0.301013 644.349775
        prior_purchase_count              3.0  0.298811 634.035975
          purchase_count_30d              4.0  0.263339 481.887008
           prior_total_spend              5.0  0.261934 476.383042
               discount_rate              6.0  0.221933 335.029275
      purchase_rate_per_year              7.0  0.210093 298.629128
         prior_mean_gap_days              8.0  0.195046 255.752741
                  spend_365d              9.0  0.183368 225.009793
days_since_previous_purchase             10.0  0.169630 191.596863
                   spend_30d             11.0  0.165181 181.400251
          prior_std_gap_days             12.0  0.153751 156.577929
           is_first_purchase             13.0  0.146232 141.310615
    prior_std_purc

### Final Justification Summary

- **Identifiers Handled:** `ID` and `customer_id` were removed to prevent index memorization and sparse cardinality expansion. `random_noise` was evaluated and discarded as all retained features exceeded noise importance thresholds.

- **Short-Term Recency Signals Retained:** Variables such as `purchase_count_30d`, `purchase_count_90d`, `spend_30d`, and `spend_90d` consistently rank at the top across ANOVA F-score and correlation metrics. Recency is the strongest signal for 90-day repurchase behavior.

- **Historical Baseline Retained:** Aggregate metrics (`prior_purchase_count`, `days_since_previous_purchase`, `customer_tenure_days`) and the structural flag is_first_purchase are kept to ensure the model distinguishes long-term repeat buyers from first-time shoppers.

- **Multicollinear Redundancy Removed:** Pairwise correlation analysis ($\vert{}r\vert{} > 0.85$) identified and eliminated redundant overlapping features, preserving model stability and interpretability.